# Workshop 1 · Gold KPI Model for Power BI

![Gold model contract](../../assets/images/w1_gold_model_contract.png)

In this workshop you build the complete RetailHub Gold model used by the optional Day1_03 checkpoint and the Day 2 Power BI handoff. This is the place where the full Kimball-style model is created; the earlier demos only prepared the concepts.


## Business Scenario

RetailHub has inconsistent revenue and margin numbers across reports. The BI team needs one governed Gold model with conformed dimensions, a line-grain fact table and a Power BI-ready dashboard table.

The source is `silver.*`. The output is `gold.*`. Do not read directly from Bronze and do not rebuild source data.


## Target Contract

By the end, the Gold schema must contain these durable objects:

| Object | Grain | Purpose |
|---|---|---|
| `gold.dim_date` | one row per calendar date | date slicers, calendar attributes, relationship target |
| `gold.dim_customer` | one row per customer | customer geography and segment slicers |
| `gold.dim_product` | one row per product | product category and active-product slicers |
| `gold.fact_sales` | one row per order line | governed measures and relationship source |
| `gold.fact_sales_dashboard` | one row per order line | wide table for fast Power BI report prototyping |

Expected observation: `fact_sales` and `fact_sales_dashboard` keep the same line grain. The dashboard table is wider, not more aggregated.


## Setup

Resolve the participant catalog. `00_setup` executes `USE CATALOG`, so SQL cells can use two-part names such as `silver.order_lines`.


In [ ]:
%run ../../setup/00_setup


### Configuration

Confirm the active catalog and schemas before running SQL cells.


In [ ]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("BRONZE", BRONZE),
    ("SILVER", SILVER),
    ("GOLD", GOLD),
], ["Variable", "Value"]))


### Runtime Pre-check

This notebook requires the four Silver source tables created by `data/generate_training_dataset.ipynb`.


In [ ]:
required_tables = [
    f"{SILVER}.customers",
    f"{SILVER}.products",
    f"{SILVER}.sales_orders",
    f"{SILVER}.order_lines",
]
missing = [t for t in required_tables if not spark.catalog.tableExists(t)]
if missing:
    for table in missing:
        print("[MISSING]", table)
    raise Exception("Pre-check failed. Run data/generate_training_dataset.ipynb first.")
print("[OK] Silver source tables are available.")


## Workshop Rubric

A correct solution must satisfy these checks:

- every standalone SQL step is a `%sql` cell,
- dimensions have unique keys,
- `gold.fact_sales` has one row per `line_id`,
- `gold.fact_sales_dashboard` keeps one row per `line_id`,
- date, customer and product relationships can be built as many-to-one relationships in Power BI,
- KPI rules are explicit: revenue and margin count completed rows only,
- unknown dimension attributes are visible through controlled labels, not silent row loss.


## Source Discovery Before Modeling

Before creating Gold, verify the source volume. This gives you a baseline for the model checks later.


In [ ]:
%sql
SELECT 'customers' AS source_table, COUNT(*) AS rows FROM silver.customers
UNION ALL
SELECT 'products', COUNT(*) FROM silver.products
UNION ALL
SELECT 'sales_orders', COUNT(*) FROM silver.sales_orders
UNION ALL
SELECT 'order_lines', COUNT(*) FROM silver.order_lines
ORDER BY source_table


In [ ]:
%sql
SELECT
  COUNT(*) AS line_rows,
  COUNT(DISTINCT line_id) AS distinct_lines,
  COUNT(DISTINCT order_id) AS distinct_orders,
  ROUND(COUNT(*) / NULLIF(COUNT(DISTINCT order_id), 0), 2) AS avg_lines_per_order,
  MIN(order_date) AS min_order_date,
  MAX(order_date) AS max_order_date
FROM silver.order_lines


## Task 1: Create Gold Schema

Create the Gold schema if it does not already exist. Keep the object name exactly `gold` because the optional checkpoint and Day 2 notebooks use that contract.


In [ ]:
%sql
-- TODO: create the Gold schema if it does not exist.
-- Acceptance: SHOW SCHEMAS LIKE 'gold' returns one row.


In [ ]:
%sql
SHOW SCHEMAS LIKE 'gold'


## Task 2: Build `gold.dim_date`

Create a date dimension from `2024-01-01` to `2026-12-31`.

Required columns: `date_key`, `year`, `quarter`, `month`, `month_name`, `year_month`, `day_of_month`, `day_of_week`, `day_name`, `is_weekend`.

Acceptance: one row per date, no duplicate `date_key`, min/max dates match the required range.


In [ ]:
%sql
-- TODO: create or replace gold.dim_date.
-- Hint: use explode(sequence(to_date('2024-01-01'), to_date('2026-12-31'), interval 1 day)).


In [ ]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT date_key) AS distinct_dates,
  COUNT(*) - COUNT(DISTINCT date_key) AS duplicate_date_keys,
  MIN(date_key) AS min_date,
  MAX(date_key) AS max_date
FROM gold.dim_date


## Task 3: Build `gold.dim_customer`

Create one row per customer from `silver.customers`.

Required columns: `customer_id`, `customer_name`, `city`, `customer_region`, `country`, `segment`, `created_date`.

Acceptance: one row per `customer_id`. Use `region AS customer_region` to make the column explicit for BI users.


In [ ]:
%sql
-- TODO: create or replace gold.dim_customer from silver.customers.
-- Required rename: region AS customer_region.


In [ ]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT customer_id) AS distinct_customers,
  COUNT(*) - COUNT(DISTINCT customer_id) AS duplicate_customer_keys
FROM gold.dim_customer


## Task 4: Build `gold.dim_product`

Create one row per product from `silver.products`.

Required columns: `product_id`, `product_name`, `category`, `subcategory`, `standard_unit_cost`, `list_unit_price`, `is_active`.

Acceptance: one row per `product_id`. Use business-friendly measure names: `unit_cost AS standard_unit_cost`, `unit_price AS list_unit_price`.


In [ ]:
%sql
-- TODO: create or replace gold.dim_product from silver.products.
-- Required renames: unit_cost AS standard_unit_cost, unit_price AS list_unit_price.


In [ ]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT product_id) AS distinct_products,
  COUNT(*) - COUNT(DISTINCT product_id) AS duplicate_product_keys
FROM gold.dim_product


## Task 5: Build `gold.fact_sales`

Create the governed line-grain fact table from `silver.order_lines`.

Required grain: one row per `line_id`.

Required keys: `line_id`, `order_id`, `order_date`, `customer_id`, `product_id`.

Required measures: `quantity`, `unit_price`, `unit_cost`, `discount_pct`, `line_revenue`, `line_cost`, `line_margin`.

Required flags: `is_completed`, `is_returned`, `is_cancelled`.


In [ ]:
%sql
-- TODO: create or replace gold.fact_sales.
-- Acceptance: duplicate_line_ids = 0 in the next check.


In [ ]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT line_id) AS distinct_lines,
  COUNT(*) - COUNT(DISTINCT line_id) AS duplicate_line_ids,
  COUNT(DISTINCT order_id) AS distinct_orders
FROM gold.fact_sales


## Task 6: Build `gold.fact_sales_dashboard`

Create a Power BI-ready wide table by joining `gold.fact_sales` to the dimensions.

Required behavior:

- keep one row per `line_id`,
- include date, customer and product attributes used by slicers,
- use `COALESCE` for missing dimension attributes so blank slicers become visible labels,
- do not aggregate rows in this table.


In [ ]:
%sql
-- TODO: create or replace gold.fact_sales_dashboard.
-- Acceptance: duplicate_lines = 0 and rows = rows in gold.fact_sales.


In [ ]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT line_id) AS distinct_lines,
  COUNT(*) - COUNT(DISTINCT line_id) AS duplicate_lines
FROM gold.fact_sales_dashboard


## Task 7: Relationship Readiness

Confirm that Power BI can create many-to-one relationships from the fact table to the dimensions.


In [ ]:
%sql
-- TODO: return one row per relationship with orphan_rows.
-- Required relationships:
-- fact_sales.order_date -> dim_date.date_key
-- fact_sales.customer_id -> dim_customer.customer_id
-- fact_sales.product_id -> dim_product.product_id


## Task 8: KPI Smoke Test

Compute the baseline KPI set from `gold.fact_sales_dashboard`.

Rules:

- Revenue: sum `line_revenue` for completed rows only.
- Gross Margin: sum `line_margin` for completed rows only.
- Completed Orders: distinct completed `order_id`.
- Margin Rate %: Gross Margin divided by Revenue, protected with `NULLIF`.


In [ ]:
%sql
-- TODO: compute revenue, gross_margin, completed_orders and margin_rate_pct from gold.fact_sales_dashboard.


## Task 9: Model Change Impact Check

Program requirement: show how a model or filter decision changes results and performance.

You will compare two things:

1. KPI filter logic: completed-only revenue vs a broader completed-or-returned scenario.
2. Query shape: star schema query vs wide dashboard-table query.

Use `TEMP VIEW` objects only. Do not change the final Gold contract.

In [ ]:
%sql
-- TODO: create TEMP VIEW w1_status_filter_comparison.
-- Required columns:
--   scenario_name
--   revenue
--   order_count
-- Required rows:
--   completed_only
--   completed_or_returned
-- Hint: use gold.fact_sales_dashboard and explicit status flags.

In [ ]:
%sql
-- TODO: create TEMP VIEW w1_model_pattern_comparison.
-- Required columns:
--   pattern_name
--   revenue
--   line_rows
-- Required rows:
--   star_schema
--   wide_dashboard
-- Rule: both rows should use completed sales only.

In [ ]:
%sql
-- Check after your TODO views exist.
SELECT
  'status_filter_delta' AS check_name,
  ROUND(
    MAX(CASE WHEN scenario_name = 'completed_or_returned' THEN revenue END)
    - MAX(CASE WHEN scenario_name = 'completed_only' THEN revenue END),
    2
  ) AS observed_delta
FROM w1_status_filter_comparison
UNION ALL
SELECT
  'star_vs_wide_revenue_delta',
  ROUND(
    MAX(CASE WHEN pattern_name = 'star_schema' THEN revenue END)
    - MAX(CASE WHEN pattern_name = 'wide_dashboard' THEN revenue END),
    2
  )
FROM w1_model_pattern_comparison

Expected observation: the status filter delta should explain a business definition change. The star-vs-wide revenue delta should be zero when both queries implement the same KPI rule.

## Task 10: Object Inventory and Handoff

Create a final object inventory for the trainer and for the optional Day1_03 checkpoint.


In [ ]:
%sql
-- TODO: return row counts for all five Gold objects.


## Completion Checklist

Before moving to the optional checkpoint or Day 2, confirm:

- all five target objects exist,
- dimensions have unique keys,
- fact and dashboard tables are line-grain,
- dashboard table includes date, customer, product, channel, status and measure columns,
- KPI smoke test returns non-null business values,
- model-change task explains filter impact and star-vs-wide consistency,
- relationship readiness check returns zero orphan rows or a documented source-quality reason.
